# 03 · Peptide embeddings from protein language models

Extracts residue-level embeddings of each 18-residue peptide (wild type and every point mutant)
with three pLMs and measures how far each mutation moves the representation
(report, section 2).

| Model | Hugging Face ID | Embedding dim. |
|---|---|---|
| ESM-2 (650M) | `facebook/esm2_t33_650M_UR50D` | 1280 |
| ProtT5-XL-UniRef50 (encoder) | `Rostlab/prot_t5_xl_uniref50` | 1024 |
| Ankh base (encoder) | `ElnaggarLab/ankh-base` | 768 |

**Context.** Each peptide is embedded *inside its full protein sequence* (N-terminal region +
peptide + C-terminal region) and only the peptide window is kept.

**Distances between a mutant and its WT** (both are matrices *peptide length × embedding dim.*):

* local distance at the mutated position `p`: $d_p = \lVert e^{mut}_p - e^{WT}_p \rVert_2$
* global distance: $D = \lVert E^{mut} - E^{WT} \rVert_F$ (Frobenius norm)
* position-wise distances $\lVert e^{mut}_j - e^{WT}_j \rVert_2$ for **every** position `j`
  (`pos_1 … pos_18`), used as regression features in notebook 06.

**Inputs:** `data/processed/df_tp.csv`, `data/processed/df_dms.csv` (notebook 01) and the protein
FASTA file `data/raw/source_proteins.fasta`.
**Outputs (`data/processed/`):** `wt_embeddings.npz`, `mutant_embedding_distances.csv`.

> **Hardware.** ProtT5-XL and ESM-2 (650M) need a GPU for reasonable run times (e.g. a free Google Colab
> GPU runtime). All other notebooks run on a laptop using the files saved here.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import gc
import re

import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, T5EncoderModel, T5Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Configuration

In [ ]:
FASTA_FILE = RAW_DIR / "source_proteins.fasta"

# `start_offset` = number of special tokens the tokenizer puts BEFORE the first residue.
#   * ESM-2 prepends <cls>                          -> 1
#   * T5-style tokenizers (ProtT5, Ankh) only append </s> at the end -> 0
# The extraction function checks the token count, so a wrong value fails loudly.
EMBEDDING_MODELS = {
    "ESM-2 (650M)": {"hf_id": "facebook/esm2_t33_650M_UR50D", "start_offset": 1, "dim": 1280},
    "ProtT5": {"hf_id": "Rostlab/prot_t5_xl_uniref50", "start_offset": 0, "dim": 1024},
    "Ankh": {"hf_id": "ElnaggarLab/ankh-base", "start_offset": 0, "dim": 768},
}

CORE_POSITIONS = (5, 7, 9)  # L, C, E of the LxCxE motif
AA_ORDER = list("HKRDENQSTCMAILVFWYGP")

## 2. Load processed data and full protein sequences

In [ ]:
df_tp = pd.read_csv(PROCESSED_DIR / "df_tp.csv")
df_dms = pd.read_csv(PROCESSED_DIR / "df_dms.csv")


def read_fasta(path):
    """Read a FASTA file into a {header_id: sequence} dictionary."""
    sequences, header = {}, None
    with open(path) as handle:
        for line in handle:
            line = line.strip()
            if line.startswith(">"):
                header = line[1:].split()[0]  # e.g. "sp|P29374|ARI4A_HUMAN"
                sequences[header] = []
            elif header is not None and line:
                sequences[header].append(line)
    return {key: "".join(parts) for key, parts in sequences.items()}


if not FASTA_FILE.exists():
    raise FileNotFoundError(f"FASTA file not found: {FASTA_FILE} (see data/README.md)")
fasta = read_fasta(FASTA_FILE)
print(f"{len(fasta)} sequences read from {FASTA_FILE.name}")

### Locate each WT peptide in its protein

The N- and C-terminal regions around the peptide give the pLMs their sequence context.

In [ ]:
wt_context = {}
for row in df_tp.itertuples():
    key = next((k for k in fasta if row.uniprot_id in k), None)
    if key is None:
        raise KeyError(f"{row.uniprot_id} ({row.protein}) is not in the FASTA file")

    start = fasta[key].find(row.peptide)
    if start == -1:
        raise ValueError(f"Peptide {row.peptide} not found in {key}")

    wt_context[row.protein] = {
        "fasta_id": key,
        "peptide": row.peptide,
        "start": start,  # 0-based position of the peptide in the protein
        "n_term": fasta[key][:start],
        "c_term": fasta[key][start + len(row.peptide):],
    }
    print(f"{row.protein:16s} {key:28s} peptide at residues {start + 1}-{start + len(row.peptide)}")

## 3. Embedding models

`load_embedding_model` returns a function `extract(full_sequence, peptide_start, peptide_length)`
that returns the peptide embedding matrix of shape *(peptide_length, embedding_dim)*.

In [ ]:
def load_embedding_model(model_name):
    """Load one pLM and return its peptide-embedding extraction function."""
    cfg = EMBEDDING_MODELS[model_name]
    hf_id, start_offset = cfg["hf_id"], cfg["start_offset"]

    if model_name.startswith("ESM-2"):
        tokenizer = AutoTokenizer.from_pretrained(hf_id)
        model = AutoModel.from_pretrained(hf_id).to(device).eval()

        def encode(sequence):
            return tokenizer(sequence, return_tensors="pt")

    elif model_name == "ProtT5":
        tokenizer = T5Tokenizer.from_pretrained(hf_id, do_lower_case=False)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = T5EncoderModel.from_pretrained(hf_id, torch_dtype=dtype).to(device).eval()

        def encode(sequence):
            # ProtT5 expects space-separated residues; rare amino acids -> X
            return tokenizer(" ".join(re.sub(r"[UZOB]", "X", sequence)), return_tensors="pt")

    elif model_name == "Ankh":
        tokenizer = AutoTokenizer.from_pretrained(hf_id)
        model = T5EncoderModel.from_pretrained(hf_id).to(device).eval()

        def encode(sequence):
            # Ankh expects a list of residues (one "word" per amino acid)
            return tokenizer(list(sequence), is_split_into_words=True, return_tensors="pt")

    else:
        raise ValueError(f"Unknown model: {model_name}")

    @torch.no_grad()
    def extract(full_sequence, peptide_start, peptide_length):
        inputs = encode(full_sequence).to(device)
        hidden = model(**inputs).last_hidden_state  # (1, n_tokens, dim)

        expected_tokens = len(full_sequence) + start_offset + 1  # + end-of-sequence token
        if hidden.shape[1] != expected_tokens:
            raise ValueError(
                f"{model_name}: got {hidden.shape[1]} tokens for {len(full_sequence)} residues "
                f"(expected {expected_tokens}). Check `start_offset` in EMBEDDING_MODELS."
            )

        first = start_offset + peptide_start
        window = hidden[0, first:first + peptide_length, :]
        return window.float().cpu().numpy()

    return extract

## 4. Mutational scan

For every model and protein:
1. embed the WT peptide in its protein context;
2. embed each mutant (same context, mutated peptide);
3. store the local, global and position-wise distances to the WT embedding.

In [ ]:
def scan_protein(protein, model_name, extract):
    """Embed the WT and all mutants of one protein; return (wt_embedding, records)."""
    ctx = wt_context[protein]
    length = len(ctx["peptide"])
    dim = EMBEDDING_MODELS[model_name]["dim"]

    wt_embedding = extract(ctx["n_term"] + ctx["peptide"] + ctx["c_term"], ctx["start"], length)
    assert wt_embedding.shape == (length, dim), wt_embedding.shape

    records = []
    mutants = df_dms[df_dms["protein"] == protein]
    for mut in tqdm(mutants.itertuples(), total=len(mutants), desc=f"{model_name} | {protein}"):
        sequence = ctx["n_term"] + mut.peptide + ctx["c_term"]
        diff = extract(sequence, ctx["start"], length) - wt_embedding  # (length, dim)

        position_distances = np.linalg.norm(diff, axis=1)  # ||e_mut,j - e_wt,j||_2 per position j
        record = {
            "protein": protein,
            "mutation": mut.mutation,
            "position": mut.position,
            "mut_aa": mut.mut_aa,
            "embedding_model": model_name,
            "dist_local": position_distances[mut.position - 1],  # d_p
            "dist_global": np.linalg.norm(diff),  # D (Frobenius norm)
        }
        record.update({f"pos_{j + 1}": d for j, d in enumerate(position_distances)})
        records.append(record)
    return wt_embedding, records

In [ ]:
wt_embeddings = {}  # (protein, model) -> array (peptide length, dim)
distance_records = []

for model_name in EMBEDDING_MODELS:
    extract = load_embedding_model(model_name)
    for protein in df_tp["protein"]:
        wt_embedding, records = scan_protein(protein, model_name, extract)
        wt_embeddings[(protein, model_name)] = wt_embedding
        distance_records.extend(records)

    # Free GPU memory before loading the next model
    del extract
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_distances = pd.DataFrame(distance_records)
df_distances.head()

## 5. Save outputs

In [ ]:
df_distances.to_csv(PROCESSED_DIR / "mutant_embedding_distances.csv", index=False)

# Embedding matrices cannot be stored faithfully in a CSV (they get truncated),
# so they are saved as a compressed NumPy archive with keys "<protein>__<model>".
np.savez_compressed(
    PROCESSED_DIR / "wt_embeddings.npz",
    **{f"{protein}__{model}": emb for (protein, model), emb in wt_embeddings.items()},
)
print(f"Saved {len(df_distances)} rows and {len(wt_embeddings)} WT embedding matrices to {PROCESSED_DIR}")

## 6. Visualisation: local vs. global mutational distances

Report figures 4 and S1. Positions 5, 7 and 9 (LxCxE core) are highlighted in red.

In [ ]:
def plot_distance_heatmaps(protein, model_name, save=True):
    """Local (d_p) and global (D) embedding distances for every point mutation."""
    sub = df_distances[
        (df_distances["protein"] == protein) & (df_distances["embedding_model"] == model_name)
    ]
    wt = wt_embeddings[(protein, model_name)]
    wt_norms = np.linalg.norm(wt, axis=1)
    wt_peptide = wt_context[protein]["peptide"]

    panels = [
        ("dist_local", f"Local distance\nWT norm range [{wt_norms.min():.2f}, {wt_norms.max():.2f}]"
                       f" | mean {wt_norms.mean():.2f}"),
        ("dist_global", f"Global distance\nWT Frobenius norm: {np.linalg.norm(wt):.2f}"),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    for ax, (value_col, title) in zip(axes, panels):
        matrix = sub.pivot(index="mut_aa", columns="position", values=value_col)
        matrix = matrix.reindex(AA_ORDER)
        sns.heatmap(matrix, ax=ax, cmap="Blues", linewidths=0.5, linecolor="whitesmoke",
                    cbar_kws={"label": "||mutant - WT||"})
        ax.xaxis.tick_top()  # must come before the tick labels are styled
        ax.set_xticklabels([f"{wt_peptide[p - 1]}\n{p}" for p in matrix.columns], rotation=0)
        for tick, pos in zip(ax.get_xticklabels(), matrix.columns):
            if pos in CORE_POSITIONS:
                tick.set_color("red")
                tick.set_fontweight("bold")
        ax.set_xlabel("")
        ax.set_ylabel("Substituted amino acid")
        ax.set_title(title, fontsize=15, pad=15, fontweight="bold")
    fig.suptitle(f"Mutational distance analysis: {protein} ({model_name})",
                 fontsize=18, fontweight="bold", y=1.03)
    plt.tight_layout()
    if save:
        file_name = re.sub(r"[^A-Za-z0-9]+", "_", f"embedding_distances_{protein}_{model_name}")
        save_figure(file_name.strip("_") + ".png")
    plt.show()


for model_name in EMBEDDING_MODELS:
    for protein in df_tp["protein"]:
        plot_distance_heatmaps(protein, model_name)